# 01 Data Collection (GĐ1)

Thu thập review tiếng Việt từ Google Play (Shopee main + Lazada external). Lưu snapshot **bất biến** vào `01_data/raw/` (JSONL).

**Protocol v1.1:** chạy **LOCAL** (Kaggle tắt internet mặc định). `lang="vi"`, `country="vn"`, `Sort.NEWEST`,
phân trang bằng `continuation_token`. **Dừng theo NGÀY**: khi review cũ nhất trong lô đã cũ hơn cutoff
**2025-10-06** (T0 = 2026-10-06 − 12 tháng); trần an toàn 120.000/app.
**Nghiệm thu:** `min(review_date) ≤ 2025-10-06` cho CẢ HAI app. Không ép hai app cùng số lượng.
Logic ở `src/collect/google_play.py` — notebook chỉ import.

In [ ]:
import sys, os, json
# Local: chạy từ notebooks/. src/ nằm trong 02_src/.
sys.path.append(os.path.abspath(os.path.join('..', '02_src')))
from src.utils.kaggle_paths import input_dir, is_kaggle
from src.utils.seeds import set_seed
from src.collect.google_play import APPS, T0, CUTOFF, preview, crawl, summarize, accept, write_manifest
set_seed(42)
assert not is_kaggle(), 'GĐ1 chạy LOCAL — Kaggle tắt internet mặc định.'
print('T0 =', T0.date(), '| cutoff =', CUTOFF.isoformat())
print(APPS)

## Bước 1 — Crawl thử ~50 review / app (KHÔNG ghi ra đĩa)
Kiểm tra: đúng app (title/developer), schema trường của thư viện, record đầu tiên đã map đúng, PII đã hash.

In [ ]:
previews = {}
for name, app_id in APPS.items():
    p = preview(app_id, n=50)
    previews[name] = p
    print(f'===== {name} ({app_id})')
    print('title     :', p['title'], '|', p['developer'], '|', p['installs'])
    print('fetched   :', p['n_fetched'], '| date range:', p['date_range'])
    print('lib fields:', p['library_fields'])
    print('record[0] :', json.dumps(p['first_record'], ensure_ascii=False, indent=2))

## Bước 2 — XÁC NHẬN trước khi crawl full
Chỉ đổi `CONFIRMED = True` sau khi người nghiên cứu đã xác nhận schema + đúng app ở Bước 1.
Chạy full crawl vào đúng **T0 = 2026-10-06** (amendment v1.3) để cửa sổ temporal khớp ngày crawl.

In [ ]:
CONFIRMED = False
assert CONFIRMED, 'Chưa xác nhận record đầu tiên → dừng, không crawl full.'

## Bước 3 — Crawl full (dừng theo ngày), ghi dần ra đĩa
File đích đã tồn tại → báo lỗi (raw bất biến). Muốn crawl lại phải dùng tên file mới và ghi amendment.

In [ ]:
runs = []
for name, app_id in APPS.items():
    out = os.path.join(input_dir('raw'), name, f'{name}_reviews_T0_{T0.date()}.jsonl')
    print(f'===== crawl {name} → {out}')
    stats = crawl(app_id, out, cutoff=CUTOFF, cap=120000, batch=200, sleep=1.0)
    summ = summarize(out)
    runs.append({'app': name, 'package_id': app_id, **stats, **summ, 'accepted': accept(summ)})
    print(json.dumps(runs[-1], ensure_ascii=False, indent=2))

## Bước 4 — Nghiệm thu GĐ1 + manifest

In [ ]:
for r in runs:
    status = '✅ ĐẠT' if r['accepted'] else '❌ CHƯA ĐẠT'
    print(f"{r['app']:7s} {status} | n={r['n_reviews']} | min={r['min_review_date']} | stop={r['stop_reason']}")
    if not r['accepted']:
        print('   → min(review_date) chưa tới cutoff. Nếu stop=safety_cap: tăng cap, crawl lại sang file mới '
              '(ghi amendment). Nếu no_more_pages/empty_batch: Google Play không trả thêm — báo người nghiên cứu.')
manifest = write_manifest(runs, os.path.join(input_dir('manifests'), f'gd1_crawl_manifest_T0_{T0.date()}.json'))
print('manifest →', os.path.join(input_dir('manifests'), f'gd1_crawl_manifest_T0_{T0.date()}.json'))

## Bước 5 — Đưa lên Kaggle
Upload `01_data/raw/{shopee,lazada}/*.jsonl` + manifest thành một Kaggle Dataset (private) cho các notebook sau
đọc qua `input_dir(...)`. KHÔNG commit raw vào git (đã có trong `.gitignore`).